In [2]:
from pathlib import Path
import pandas as pd

CSV = Path("..") / "data" / "KaggleV2-May-2016.csv"
assert CSV.exists(), f"no encuentro {CSV.resolve()} — abre el notebook desde ml/notebooks"

df = pd.read_csv(CSV)
df = df.drop(columns=["PatientId", "AppointmentID"])          # IDs = fuga
df = df[(df["Age"] >= 0) & (df["Age"] <= 110)]                # -1 / 115+ son ruido del dataset

df["scheduled"] = pd.to_datetime(df["ScheduledDay"])
df["appt"] = pd.to_datetime(df["AppointmentDay"])
df["lead_days"] = (df["appt"].dt.normalize() - df["scheduled"].dt.normalize()).dt.days
df["no_show"] = (df["No-show"] == "Yes").astype(int)

# ScheduledDay/AppointmentDay son strings: sklearn no los acepta y la info ya está en lead_days
X = df.drop(columns=["No-show", "no_show", "scheduled", "appt", "ScheduledDay", "AppointmentDay"])
y = df["no_show"]

assert "no_show" not in X.columns, "fuga de etiqueta"
assert set(y.unique()) == {0, 1}
print(f"{len(df):,} filas · tasa no-show {y.mean():.1%}")
# Tres correcciones sobre tu snippet: no_show quedaba dentro de X (fuga de etiqueta), ScheduledDay/AppointmentDay quedaban como strings, y tr/te se calculaban pero nunca se usaban.

110,521 filas · tasa no-show 20.2%


In [3]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, roc_auc_score
from sklearn.preprocessing import OneHotEncoder

cut = df["appt"].quantile(0.8)
tr, te = df["appt"] < cut, df["appt"] >= cut

enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
num_tr = X[tr].drop(columns=["Gender", "Neighbourhood"]).to_numpy()
num_te = X[te].drop(columns=["Gender", "Neighbourhood"]).to_numpy()
cat_tr = enc.fit_transform(X[tr][["Gender", "Neighbourhood"]])
cat_te = enc.transform(X[te][["Gender", "Neighbourhood"]])

print(f"corte: {cut.date()} · train {tr.sum():,} · test {te.sum():,}")
print(f"baseline (predecir todo 'asiste'): recall no-show = 0.00 · prior = {y[te].mean():.1%}")

model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(np.hstack([num_tr, cat_tr]), y[tr])
proba = model.predict_proba(np.hstack([num_te, cat_te]))[:, 1]

for t in (0.5, 0.35, 0.25):
    pred = (proba >= t).astype(int)
    print(
        f"umbral {t}: recall={recall_score(y[te], pred):.2f} "
        f"precision={precision_score(y[te], pred, zero_division=0):.2f} "
        f"auc={roc_auc_score(y[te], proba):.3f}"
    )

corte: 2016-06-01 · train 84,072 · test 26,449
baseline (predecir todo 'asiste'): recall no-show = 0.00 · prior = 18.5%
umbral 0.5: recall=0.63 precision=0.28 auc=0.665
umbral 0.35: recall=0.93 precision=0.20 auc=0.665
umbral 0.25: recall=1.00 precision=0.19 auc=0.665


In [4]:
assert roc_auc_score(y[te], proba) > 0.55, "el modelo no supera el azar: revisa features"
assert recall_score(y[te], (proba >= 0.5).astype(int)) > 0.0, "nunca predice no-show"
print("OK")

OK
